# 01 — Data Loading & Quality Checks

Load the raw export and check it the way you'd check any dataset before trusting it: shape, types, missing values, duplicates, invalid values, and date integrity.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

df = pd.read_csv("../data/sales_raw.csv")
df.shape

(36535, 10)

In [2]:
df.head(10)

         date product_id        category  ...    store  region  holiday
0  2024-04-30       P003         Apparel  ...  STORE_5    West        0
1  2023-02-01       P030          Sports  ...  STORE_2   North        0
2  2023-03-25       P012  Home & Kitchen  ...  STORE_3   North        0
3  2023-10-29       P006     Electronics  ...  STORE_2   South        0
4  2024-07-16       P007  Home & Kitchen  ...  STORE_1   South        0
5  2025-02-13       P021     Electronics  ...  STORE_4   North        0
6  2025-06-13       P021     Electronics  ...  STORE_2   South        0
7  2024-05-26       P014          Beauty  ...  STORE_1   North        0
8  2024-03-23       P006     Electronics  ...  STORE_4   South        0
9  2024-03-16       P038         Apparel  ...  STORE_1   South        0

[10 rows x 10 columns]

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36535 entries, 0 to 36534
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        36530 non-null  str    
 1   product_id  36535 non-null  str    
 2   category    36535 non-null  str    
 3   quantity    36463 non-null  float64
 4   price       36462 non-null  float64
 5   promotion   36535 non-null  int64  
 6   discount    36535 non-null  float64
 7   store       36535 non-null  str    
 8   region      36535 non-null  str    
 9   holiday     36535 non-null  int64  
dtypes: float64(3), int64(2), str(5)
memory usage: 2.8 MB


In [4]:
df.describe(include="all").T

              count unique             top  freq  ...    25%     50%     75%     max
date          36530    914      2023-02-02    42  ...    NaN     NaN     NaN     NaN
product_id    36535     40            P023   916  ...    NaN     NaN     NaN     NaN
category      36535      5  Home & Kitchen  7310  ...    NaN     NaN     NaN     NaN
quantity    36463.0    NaN             NaN   NaN  ...   39.0    66.0    97.0   509.0
price       36462.0    NaN             NaN   NaN  ...  59.93  116.75  198.44  242.91
promotion   36535.0    NaN             NaN   NaN  ...    0.0     0.0     0.0     1.0
discount    36535.0    NaN             NaN   NaN  ...    0.0     0.0     0.0     0.3
store         36535      5         STORE_4  7548  ...    NaN     NaN     NaN     NaN
region        36535      4           North  9229  ...    NaN     NaN     NaN     NaN
holiday     36535.0    NaN             NaN   NaN  ...    0.0     0.0     0.0     1.0

[10 rows x 11 columns]

## Missing values

In [5]:
df.isna().sum().sort_values(ascending=False)

price         73
quantity      72
date           5
product_id     0
category       0
promotion      0
discount       0
store          0
region         0
holiday        0
dtype: int64

## Duplicate rows

We check for exact duplicate rows, and separately for duplicate `(date, product_id, store, region)` keys, which would mean two conflicting readings for what should be one observation.

In [6]:
print("Exact duplicate rows:", df.duplicated().sum())

key_cols = ["date", "product_id", "store", "region"]
dup_keys = df.duplicated(subset=key_cols).sum()
print("Duplicate (date, product_id, store, region) keys:", dup_keys)

Exact duplicate rows: 55
Duplicate (date, product_id, store, region) keys: 55


## Invalid numeric values

In [7]:
print("Rows with quantity < 0:", (df["quantity"] < 0).sum())
print("Rows with price < 0:   ", (df["price"] < 0).sum())
print("Rows with quantity == 0:", (df["quantity"] == 0).sum())

Rows with quantity < 0: 19
Rows with price < 0:    18
Rows with quantity == 0: 2


## Date problems

Dates arrived as strings from a CSV export, so we check for unparsable values, out-of-range/future dates, and how many rows we'd lose if we dropped them.

In [8]:
parsed = pd.to_datetime(df["date"], errors="coerce", format="%Y-%m-%d")
n_missing_date = df["date"].isna().sum()
n_unparsable = parsed.isna().sum() - n_missing_date
print("Missing date strings:      ", n_missing_date)
print("Unparsable date strings:   ", n_unparsable)
print("Min parsed date:           ", parsed.min())
print("Max parsed date:           ", parsed.max())

DATASET_END = pd.Timestamp("2025-06-30")
n_future = (parsed > DATASET_END).sum()
print("Dates beyond expected window (>2025-06-30):", n_future)

Missing date strings:       5
Unparsable date strings:    10
Min parsed date:            2023-01-01 00:00:00
Max parsed date:            2026-01-15 00:00:00
Dates beyond expected window (>2025-06-30): 10


## Summary of findings

| Check | Result | Action |
|---|---|---|
| Missing `quantity` | small % of rows | drop (can't impute a target reliably) |
| Missing `price` | small % of rows | impute with product-level median price |
| Missing `date` | ~10 rows | drop (unusable without a date) |
| Negative `quantity` / `price` | a few dozen rows | drop as data-entry errors |
| Unparsable date (`2025-13-40`) | ~10 rows | drop |
| Future date (beyond dataset window) | ~10 rows | drop |
| Exact duplicate rows | small % | drop duplicates |

None of these issues are large enough to threaten the dataset's usability,
but every one of them would silently corrupt a naive `pd.read_csv` -> model
pipeline, so we clean explicitly rather than ignore.

In [9]:
clean = df.copy()
clean["date"] = pd.to_datetime(clean["date"], errors="coerce", format="%Y-%m-%d")

before = len(clean)

# drop duplicate rows
clean = clean.drop_duplicates()

# drop rows with missing/invalid dates or missing target
clean = clean.dropna(subset=["date"])
clean = clean[clean["date"] <= DATASET_END]
clean = clean.dropna(subset=["quantity"])

# drop negative quantity/price (data-entry errors)
clean = clean[clean["quantity"] >= 0]
clean = clean[(clean["price"] >= 0) | (clean["price"].isna())]

# impute missing price with the product's median price
clean["price"] = clean.groupby("product_id")["price"].transform(lambda s: s.fillna(s.median()))

clean["quantity"] = clean["quantity"].astype(int)
clean = clean.sort_values(["product_id", "date"]).reset_index(drop=True)

after = len(clean)
print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {after}")
print(f"Rows removed:         {before - after} ({(before-after)/before:.2%})")
clean.isna().sum()

Rows before cleaning: 36535
Rows after cleaning:  36346
Rows removed:         189 (0.52%)


date          0
product_id    0
category      0
quantity      0
price         0
promotion     0
discount      0
store         0
region        0
holiday       0
dtype: int64

In [10]:
clean.to_csv("../data/sales_clean.csv", index=False)
clean.head()

        date product_id     category  ...    store  region  holiday
0 2023-01-01       P001  Electronics  ...  STORE_5   South        1
1 2023-01-02       P001  Electronics  ...  STORE_3   North        0
2 2023-01-03       P001  Electronics  ...  STORE_2    East        0
3 2023-01-04       P001  Electronics  ...  STORE_3   South        0
4 2023-01-05       P001  Electronics  ...  STORE_5   North        0

[5 rows x 10 columns]